In [27]:
from pathlib import Path
from collections import Counter
from datetime import datetime, time
import pandas as pd

DATA_DIR = Path(Path.cwd())
COUNTRY_FILES = {"US": DATA_DIR / "data/USvideos.csv", "GB": DATA_DIR / "data/GBvideos.csv", "IN": DATA_DIR / "data/INvideos.csv"}
CATEGORY_FILE = DATA_DIR / "data/categories.csv"

VIRAL_VIEW_THRESHOLD = 1_000_000
STEADY_VIEW_THRESHOLD = 100_000
DIVISIVE_LIKE_RATIO_THRESHOLD = 0.70
STEADY_LIKE_RATIO_THRESHOLD = 0.85
MIN_RATINGS_FOR_RATIO = 100

run_started_at = datetime.now().astimezone()
print(f"Run started at {run_started_at:%Y-%m-%d %H:%M:%S %Z}")
print(f"Countries {', '.join(COUNTRY_FILES)}")
print(f"Data folder: {DATA_DIR.resolve()}")
print(Path.cwd())

Run started at 2026-10-28 16:32:32 India Standard Time
Countries US, GB, IN
Data folder: D:\Python\AIML\Practice\module_2\trendpulse\part1_python_foundations
d:\Python\AIML\Practice\module_2\trendpulse\part1_python_foundations


In [38]:
REQUIRED_COLUMNS = {
    "video_id", "trending_date", "title", "channel_title", "category_id",
    "publish_time", "views", "likes", "dislikes", "comment_count",
    "comments_disabled", "ratings_disabled", "video_error_or_removed"
}

def load_country(path, country, max_rows=None):
    df = pd.read_csv(path)
    missing_columns = REQUIRED_COLUMNS.difference(df.columns)
    if missing_columns:
        raise ValueError(f"some required columns are missing: {missing_columns}")
    df['country'] = country
    records = df.to_dict('records')
    print(f"loaded from {country} {len(records)} records in path {Path(path).name}")
    return records

all_records = []
records_by_country = {}
for country, path in COUNTRY_FILES.items():
    print(path)
    country_records = load_country(path, country)  
    records_by_country[country] = country_records
    all_records.extend(country_records)
print(f"total records is {len(all_records):,}")    
print(all_records[1:5])

d:\Python\AIML\Practice\module_2\trendpulse\part1_python_foundations\data\USvideos.csv
loaded from US 16080 records in path USvideos.csv
d:\Python\AIML\Practice\module_2\trendpulse\part1_python_foundations\data\GBvideos.csv
loaded from GB 15075 records in path GBvideos.csv
d:\Python\AIML\Practice\module_2\trendpulse\part1_python_foundations\data\INvideos.csv
loaded from IN 15075 records in path INvideos.csv
total records is 46,230
[{'video_id': 'Ugyql2NlPAd', 'trending_date': '17.14.11', 'title': 'Surprising Cricket Review', 'channel_title': 'IncredibleChallenge41', 'category_id': 25, 'publish_time': '2017-11-12T22:35:48.000Z', 'tags': 'Winter', 'views': 198385, 'likes': 1159.0, 'dislikes': 492, 'comment_count': 1446.0, 'thumbnail_link': 'https://i.ytimg.com/vi/Ugyql2NlPAd/default.jpg', 'comments_disabled': False, 'ratings_disabled': False, 'video_error_or_removed': False, 'description': 'Watch this video. Subscribe for more content about bangalore. Business enquiries in the channel ab

In [36]:
category_df = pd.read_csv(CATEGORY_FILE)
category_records = category_df.to_dict('records')

category_names = {}
for row in category_records:
    try:
        category_names[int(row['category_id'])] = str(row['category_name'])
    except (ValueError, TypeError, KeyError):
        continue  

print(f"total category records: {len(category_names)}")
print(f"Example lookup: {category_names.get(10, "Unknown category")}")    

total category records: 17
Example lookup: Music


In [44]:
NUMERIC_FIELDS = ["views", "likes", "dislikes", "comment_count"]
BOOLEAN_FIELDS = ["comments_disabled", "ratings_disabled", "video_error_or_removed"]
PARSE_FAILURES = {field: 0 for field in NUMERIC_FIELDS + BOOLEAN_FIELDS + ["trending_date", "publish_time"]}

def to_nonnegative_int(value, field_name):
    try:    
        if value is None or (isinstance(value, float) and value != value) or str(value).strip() == "":
            raise ValueError("Blank numeric value")
        number = int(value)
        if number < 0:
            raise ValueError("Number is negative")
        return number
    except(TypeError, ValueError, OverflowError):
        PARSE_FAILURES[field_name] += 1
        return 0

def to_bool(value, field_name):
    if isinstance(value, bool):
        return value
    if value is None or (isinstance(value, float) and value != value):
        PARSE_FAILURES[field_name] += 1
        return False
    normalized = str(value).strip().lower()
    print(normalized,"norm")
    if normalized in {"true": "1", "yes": "y"}:   
        return True
    if normalized in {"false": "0", "no": "n"}:
        return False
    PARSE_FAILURES[field_name] += 1
    return False

def parse_trending_date(value):
    try:
        return datetime.strptime(str(value).strip(), "%y.%d.%m").date()
    except (ValueError, TypeError):
        PARSE_FAILURES["trending_time"] += 1
        return None
    
def parse_publish_time(value):
    try:
        raw = str(value).strip()
        if raw.endswith("Z"):
            raw = raw[:-1]
        return datetime.fromisoformat(raw)
    except (ValueError, TypeError):
            PARSE_FAILURES["publish_time"] += 1
            return None

cleaned_records = []
for raw_record in all_records:
    record = dict(raw_record)
    for field in NUMERIC_FIELDS:
        record[field] = to_nonnegative_int(record.get(field), field)
    for field in BOOLEAN_FIELDS:
         record[field] = to_bool(record.get(field),field)    
    record["trending_date"] = parse_trending_date(record.get("trending_date"))   
    record["publish_time"] = parse_publish_time(record.get("publish_time")) 
    try:
        record["category_id"] = int(record["category_id"])
    except (ValueError, TypeError, KeyError):
        record["category_id"] = None

    try:
        record["category_name"] = category_names.get('category_id', "Unknown category")
    except (ValueError, TypeError, KeyError):
        record["category_name"] = None
    cleaned_records.append(record)

print(f"records length: {len(cleaned_records)}")
print("Parse-failure counts (includes blanks and invalid values):")
for field, count in PARSE_FAILURES.items():
    print(f"{field}, {count}")

records length: 46230
Parse-failure counts (includes blanks and invalid values):
views, 0
likes, 12
dislikes, 0
comment_count, 36
comments_disabled, 0
ratings_disabled, 0
video_error_or_removed, 0
trending_date, 0
publish_time, 0


In [ ]:
video_sanpshots = []
for record in cleaned_records:
    video_sanpshots.append((record["video_id"], record["trending_date"], record["views"], record["likes"]))

print("Example tuples:")
print(video_sanpshots[0])
print(f"Tuple immutable: {isinstance(video_sanpshots[0], tuple)}")  

video_ids_by_country = {}

for country, records in records_by_country.items():
    ids = set()
    for record in records:
        video_id = record.get("video_id")
        if video_id:
            ids.add(str(video_id))
    video_ids_by_country[country] = ids
    print(f"{country}: {len(ids):,} unique video IDs")

Example tuples:
('g_N-w8LeAi8', datetime.date(2017, 11, 14), 48555, 2189)
Tuple immutable: True
